# Machine Learning Lab Exam — Reusable Cheatsheet (Generic Dataset)

This notebook is **dataset-agnostic**: it works for any **CSV classification** dataset you are given in the exam.

What you’ll edit in the exam:
- `CSV_PATH` → filename of the CSV you’re given
- `TARGET_COL` → name of the target/label column (the thing you’re predicting)
- `DROP_COLS` → any columns you should exclude (IDs, names, leakage columns, free-text, etc.)
- `POS_LABEL` (optional) → for F1 when the “positive” class isn’t `1`

> **Random seed rule**: set `SEED = <your student ID number>` (remove leading `G` and any leading zeros).

---

## How to adapt when the CSV changes
1. Put the exam CSV in the same folder as this notebook.
2. Change `CSV_PATH` and `TARGET_COL`.
3. If the dataset has categorical columns (strings), this notebook handles them automatically using one-hot encoding.
4. If there are missing values or “fake zeros” used to represent missing values, define them in `ZERO_AS_MISSING_COLS`.

---


In [ ]:
# Core imports
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Display
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


## 0) Exam variables 

- `CSV_PATH`: the filename you are given (e.g. `"exam_dataset.csv"`)
- `TARGET_COL`: label column name
- `DROP_COLS`: columns to remove from features (IDs, names, leakage)
- `ZERO_AS_MISSING_COLS`: columns where 0 should be treated as missing (optional)


In [ ]:
# === EDIT THESE IN THE EXAM ===
SEED = 432121  # <-- replace with your student-number seed (no G, no leading zeros)
CSV_PATH = "data.csv"  # <-- replace with exam CSV filename
TARGET_COL = "target"  # <-- replace with label column name

DROP_COLS = []  # e.g. ["Name", "ID", "Total"] if not appropriate
ZERO_AS_MISSING_COLS = []  # e.g. ["Glucose", "BloodPressure"] when 0 means missing
POS_LABEL = 1  # used for F1. Change if your positive class isn't 1.


## 1) Load data + quick checks (Task 1.1 / 1.2)

If your CSV is elsewhere, update `CSV_PATH` accordingly.

Common fix:
- If you see `FileNotFoundError`, the CSV isn’t in the same directory as the notebook. Check the file list and adjust the path.


In [ ]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()


In [ ]:
# Missing values (NaN) per column
df.isna().sum().sort_values(ascending=False).head(30)


In [ ]:
# Optional: treat zeros as missing for specified columns
df_clean = df.copy()
for c in ZERO_AS_MISSING_COLS:
    if c in df_clean.columns:
        df_clean.loc[df_clean[c] == 0, c] = np.nan

# Re-check missing values after the replacement
df_clean.isna().sum().sort_values(ascending=False).head(30)


## 1.3) Class distribution plot (Task 1.3)

Use this to comment on class imbalance (e.g., 90/10 split).


In [ ]:
target_counts = df_clean[TARGET_COL].value_counts(dropna=False)
print(target_counts)

ax = target_counts.plot(kind="bar")
ax.set_title(f"Class Distribution: {TARGET_COL}")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.show()


## 1.4) Feature distributions / boxplots (Task 1.4)

Pick **at least 4** useful numeric features to compare across classes.
If you’re unsure, start with the first few numeric columns.


In [ ]:
# Auto-pick numeric columns (excluding target) for quick plotting
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_COL in num_cols:
    num_cols.remove(TARGET_COL)

print("Numeric columns:", num_cols[:20], "..." if len(num_cols) > 20 else "")

# Choose up to 6 to view quickly (edit the list if needed)
PLOT_COLS = num_cols[:6]
PLOT_COLS


In [ ]:
# Boxplots per class for selected features
for col in PLOT_COLS:
    if col == TARGET_COL: 
        continue
    df_clean.boxplot(column=col, by=TARGET_COL)
    plt.title(f"{col} by {TARGET_COL}")
    plt.suptitle("")
    plt.xlabel(TARGET_COL)
    plt.ylabel(col)
    plt.show()


### Task 1 summary (write 3–5 sentences)

Talk about:
- dataset size + class balance
- missing values / quality issues (NaNs, impossible zeros, outliers)
- which features look most discriminative


## 2) Data preparation + 60/20/20 split (Task 2)

We split as:
- 60% train
- 20% validation
- 20% test

Using stratification on the target.


In [ ]:
# Drop target + any non-feature columns
df_model = df_clean.drop(columns=DROP_COLS, errors="ignore").copy()

X = df_model.drop(columns=[TARGET_COL])
y = df_model[TARGET_COL]

# 60/20/20: first split off 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# then split trainval into 60% train and 20% val -> val is 0.25 of trainval (because trainval is 80%)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=SEED, stratify=y_trainval
)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)


## 2.1) Preprocessing pipeline (handles numeric + categorical)

- Numeric: impute missing → scale
- Categorical: impute missing → one-hot encode

This makes models robust across different CSVs.


In [ ]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

numeric_features, categorical_features


## 3) Model training + cross-validated hyperparameter tuning (Task 3)

Pick **three different types** from:
- kNN
- Logistic Regression
- SVM
- Random Forest

Below: templates for all 4. Use any 3.

Notes:
- Use `StratifiedKFold` to keep class balance in each fold.
- Tune **at least 2 hyperparameters** per model.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def run_grid(model, param_grid, model_name):
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    grid = GridSearchCV(
        pipe,
        param_grid=param_grid,
        scoring="f1",            # F1 is usually better when classes are imbalanced
        cv=cv,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"=== {model_name} ===")
    print("Best params:", grid.best_params_)
    print("Best mean CV score (F1):", grid.best_score_)
    return grid


### Model A: k-Nearest Neighbours (example grid)

Why reasonable:
- simple baseline, can work well when boundaries are smooth
- benefits from scaling (already included)


In [ ]:
knn_grid = {
    "model__n_neighbors": [3, 5, 7, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],  # 1=Manhattan, 2=Euclidean
}
knn_best = run_grid(KNeighborsClassifier(), knn_grid, "kNN")


### Model B: Logistic Regression (example grid)

Why reasonable:
- strong linear baseline, fast, interpretable
- works well with scaling / one-hot


In [ ]:
lr_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs", "liblinear"],
    "model__class_weight": [None, "balanced"],
    "model__max_iter": [2000],
}
lr_best = run_grid(LogisticRegression(), lr_grid, "Logistic Regression")


### Model C: Support Vector Machine (example grid)

Why reasonable:
- effective for medium-sized datasets
- RBF kernel can model non-linear boundaries


In [ ]:
svm_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", "auto"],
    "model__kernel": ["rbf", "linear"],
    "model__class_weight": [None, "balanced"],
}
svm_best = run_grid(SVC(), svm_grid, "SVM")


### Optional: Random Forest (use instead of one of the above)

Why reasonable:
- handles non-linearities & interactions
- less sensitive to scaling (but safe to keep)


In [ ]:
rf_grid = {
    "model__n_estimators": [200, 400, 800],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__class_weight": [None, "balanced"],
    "model__random_state": [SEED],
}
rf_best = run_grid(RandomForestClassifier(), rf_grid, "Random Forest")


## 4) Model comparison on validation set (Task 4)

Evaluate your **best 3** models on the validation set with:
- Accuracy
- F1

> If classes are imbalanced, **F1** is often more informative than accuracy.


In [ ]:
def eval_on_val(grid, name):
    y_pred = grid.predict(X_val)
    return {
        "model": name,
        "accuracy": accuracy_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred, pos_label=POS_LABEL),
    }

# Pick the three you actually used (edit this list in the exam)
models_used = [
    ("kNN", knn_best),
    ("LogReg", lr_best),
    ("SVM", svm_best),
    # ("RF", rf_best),
]

results = pd.DataFrame([eval_on_val(g, n) for (n, g) in models_used]).sort_values("f1", ascending=False)
results


### Task 4 interpretation (3–5 sentences)

Write about:
- which model wins on **validation**
- whether accuracy vs F1 changes the conclusion
- dataset characteristics (size, imbalance, feature types) that explain the result


## 5) Final model selection + test evaluation (Task 5)

1. Choose the best model based on validation performance (usually best F1).
2. Retrain on **train+val** with the best hyperparameters.
3. Evaluate on **test**: classification report + confusion matrix.


In [ ]:
# Select best model from validation results
best_name = results.iloc[0]["model"]
best_grid = dict(models_used)[best_name]
best_name, best_grid.best_params_


In [ ]:
# Refit on combined train+val
X_train_plus_val = pd.concat([X_train, X_val], axis=0)
y_train_plus_val = pd.concat([y_train, y_val], axis=0)

final_model = best_grid.best_estimator_
final_model.fit(X_train_plus_val, y_train_plus_val)

# Test evaluation
y_test_pred = final_model.predict(X_test)

print("Final model:", best_name)
print("\nClassification report (test):\n")
print(classification_report(y_test, y_test_pred))

cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title(f"Confusion Matrix — {best_name} (Test Set)")
plt.show()


### Task 5 interpretation (3–5 sentences)

Talk about:
- overall performance
- where it makes mistakes (which class gets confused)
- impact of class imbalance
- suitability for deployment (risk of false positives/false negatives, domain context)
